# Week 7: Deep Learning Foundations
## Notebook 12: Normalization Mechanics: BatchNorm vs LayerNorm vs GroupNorm

### 1. Conceptual Intuition & Theoretical Background
Deep neural networks suffer from **Internal Covariate Shift**: shifts in intermediate layer activation distributions that slow training.

**1. Batch Normalization (Ioffe & Szegedy, 2015):**
Normalizes across the **batch dimension** ($N$) for each feature channel ($C$) independently:
$$\mu_c = \frac{1}{N} \sum_{i=1}^{N} x_{i, c}, \quad \sigma_c^2 = \frac{1}{N} \sum_{i=1}^{N} (x_{i, c} - \mu_c)^2$$
- **Flaws:** Fails when batch size $N$ is small ($N < 8$) and breaks during sequence modeling/RNNs/Transformers due to variable sequence lengths.

**2. Layer Normalization (Ba, Kiros, Hinton, 2016):**
Normalizes across **channels and hidden features** ($C$) for each sample ($i$) independently:
$$\mu_i = \frac{1}{C} \sum_{c=1}^{C} x_{i, c}, \quad \sigma_i^2 = \frac{1}{C} \sum_{c=1}^{C} (x_{i, c} - \mu_i)^2$$
- **Advantage:** Batch-size independent; identical execution during training and inference. Standard in modern Transformers.

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} \cdot \gamma + \beta$$

---

### 2. Scratch Python Implementation: BatchNorm vs LayerNorm

In [ ]:
import numpy as np

class LayerNormScratch:
    def __init__(self, hidden_dim, eps=1e-5):
        self.eps = eps
        # Learnable affine scale (gamma) and shift (beta)
        self.gamma = np.ones(hidden_dim)
        self.beta = np.zeros(hidden_dim)
        
    def forward(self, x):
        """x: shape (batch_size, hidden_dim)"""
        # Calculate mean and variance across hidden_dim (axis 1)
        mean = np.mean(x, axis=-1, keepdims=True)
        var = np.var(x, axis=-1, keepdims=True)
        
        # Standardize
        x_hat = (x - mean) / np.sqrt(var + self.eps)
        # Scale and shift
        return self.gamma * x_hat + self.beta

class BatchNorm1DScratch:
    def __init__(self, feature_dim, momentum=0.1, eps=1e-5):
        self.eps = eps
        self.momentum = momentum
        self.gamma = np.ones(feature_dim)
        self.beta = np.zeros(feature_dim)
        # Running statistics for inference
        self.running_mean = np.zeros(feature_dim)
        self.running_var = np.ones(feature_dim)
        
    def forward(self, x, training=True):
        """x: shape (batch_size, feature_dim)"""
        if training:
            # Calculate across batch dimension (axis 0)
            mean = np.mean(x, axis=0)
            var = np.var(x, axis=0)
            
            # Update exponential moving averages
            self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * mean
            self.running_var = (1 - self.momentum) * self.running_var + self.momentum * var
            
            x_hat = (x - mean) / np.sqrt(var + self.eps)
        else:
            # Use running statistics during inference
            x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
            
        return self.gamma * x_hat + self.beta

# Compare LayerNorm vs BatchNorm on Small Batch (N=2, Dim=4)
np.random.seed(42)
small_batch = np.array([
    [10.0, 20.0, 30.0, 40.0],
    [1.0, 2.0, 3.0, 4.0]
])

ln = LayerNormScratch(hidden_dim=4)
bn = BatchNorm1DScratch(feature_dim=4)

ln_out = ln.forward(small_batch)
bn_out = bn.forward(small_batch, training=True)

print("--- Normalization Dimensional Comparison ---")
print("Input Activations (Note 10x scale disparity between sample 0 and 1):")
print(small_batch)
print("\nLayerNorm Output (Each row normalized independently to mean=0, var=1):")
print(np.round(ln_out, 3))
print(f"Row 0 Mean: {ln_out[0].mean():.1e} | Row 1 Mean: {ln_out[1].mean():.1e}")
print("\nBatchNorm Output (Features normalized across columns):")
print(np.round(bn_out, 3))

### 3. Step-by-Step Code Walkthrough
1. **LayerNorm Axis:** Normalizes across `axis=-1` (features), ensuring complete batch independence.
2. **BatchNorm Axis:** Normalizes across `axis=0` (batch), tracking running statistics for inference.
3. **Inference Consistency:** LayerNorm behaves identically during training and deployment.

---

### 4. Industry Applications & Trade-offs
- **Transformers (GPT, LLaMA, BERT):** Standardize on LayerNorm or RMSNorm because sequence lengths vary dynamically.
- **Trade-off:** BatchNorm acts as a regularizer due to batch noise, which LayerNorm lacks.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Batch Normalization fail in Transformer architectures?**
> **Answer:** In NLP, sentence lengths vary dynamically within batches. Computing statistics across tokens at padding positions corrupts the mean and variance. Furthermore, small batch sizes during autoregressive decoding degrade BatchNorm running statistics.

**Q2: What is RMSNorm and why do modern LLMs (LLaMA-3, Mistral) prefer it over LayerNorm?**
> **Answer:** RMSNorm regularizes inputs by their Root Mean Square without subtracting the mean ($\text{RMS}(x) = \sqrt{\frac{1}{d} \sum x_i^2}$). This provides identical training stability while cutting computation time by $\approx 20\%$.